In [35]:
import os
import json
import time
import pandas as pd

from pathlib import Path
from google import genai


# ============================================================
# PROJECT PATHS
# ============================================================

PROJECT_DIR = Path(
    r"C:\Users\anshu\OneDrive\Desktop\prog\Projects\reconpilot"
)

OUTPUT_DIR = PROJECT_DIR / "Tests"

AI_OUTPUT_PATH = (
    OUTPUT_DIR / "ai_investigation_results.csv"
)


# ============================================================
# GEMINI CLIENT
# ============================================================

client = genai.Client(
    api_key=os.getenv("GEMINI_API_KEY")
)


# ============================================================
# LOAD EXCEPTIONS
# ============================================================

def load_exceptions():

    results = pd.read_csv(
        OUTPUT_DIR / "reconciliation_results.csv"
    )

    exceptions = results[
        results["status"] == "EXCEPTION"
    ].copy()

    return exceptions


# ============================================================
# CLEAN VALUES
# ============================================================

def clean_value(value):

    if pd.isna(value):
        return None

    return value


# ============================================================
# BUILD CASE
# ============================================================

def build_case(row):

    case = {

        "payment_id": clean_value(
            row["payment_id"]
        ),

        "exception": clean_value(
            row["reason"]
        ),

        "payment": {

            "gross_amount": clean_value(
                row["gross_amount"]
            ),

            "payment_date": clean_value(
                row["payment_date"]
            )
        },

        "settlement": {

            "settlement_id": clean_value(
                row["settlement_id"]
            ),

            "settlement_date": clean_value(
                row["settlement_date"]
            ),

            "fee": clean_value(
                row["fee"]
            ),

            "tax": clean_value(
                row["tax"]
            ),

            "expected_net_amount": clean_value(
                row["expected_net_amount"]
            ),

            "settlement_net_amount": clean_value(
                row["settlement_net_amount"]
            ),

            "delay_days": clean_value(
                row["delay_days"]
            )
        },

        "bank": {

            "amount": clean_value(
                row["bank_amount"]
            )
        },

        "refund": {

            "amount": clean_value(
                row["refund_amount"]
            )
        }
    }

    return case


# ============================================================
# AI INVESTIGATOR
# ============================================================

def investigate(case):

    prompt = f"""
You are an AI finance operations investigator.

Your job is to investigate a financial reconciliation exception.

Use ONLY the financial evidence provided below.

Do not invent missing transactions, explanations,
or financial facts.

Determine:

1. The most likely classification of the exception.
2. Why the exception occurred based on the evidence.
3. The monetary difference if one exists.
4. What action the finance operations team should take.
5. Your confidence from 0 to 1.

Possible classifications:

- SETTLEMENT_DELAY
- MISSING_BANK_RECORD
- PARTIAL_REFUND
- UNEXPLAINED_MISMATCH

Possible actions:

- NO_ACTION
- WAIT_FOR_SETTLEMENT
- INVESTIGATE_REFUND
- ESCALATE

Rules:

- Use only the supplied evidence.
- Do not invent missing information.
- If there is no monetary discrepancy, return difference as 0.
- Confidence must be between 0 and 1.
- Explain the reasoning clearly.

Financial case:

{json.dumps(case, indent=4, default=str)}
"""

    response = client.models.generate_content(

        model="gemini-3.5-flash-lite",

        contents=prompt,

        config={

            "response_mime_type": "application/json",

            "response_schema": {

                "type": "object",

                "properties": {

                    "classification": {
                        "type": "string"
                    },

                    "explanation": {
                        "type": "string"
                    },

                    "difference": {
                        "type": "number"
                    },

                    "recommended_action": {
                        "type": "string"
                    },

                    "confidence": {
                        "type": "number"
                    }
                },

                "required": [

                    "classification",

                    "explanation",

                    "difference",

                    "recommended_action",

                    "confidence"
                ]
            }
        }
    )

    return json.loads(response.text)


# ============================================================
# LOAD PREVIOUS RESULTS
# ============================================================

def load_previous_results():

    if AI_OUTPUT_PATH.exists():

        previous = pd.read_csv(
            AI_OUTPUT_PATH
        )

        return previous

    return pd.DataFrame(
        columns=[
            "payment_id",
            "exception",
            "ai_classification",
            "explanation",
            "difference",
            "recommended_action",
            "confidence"
        ]
    )


# ============================================================
# SAVE RESULT
# ============================================================

def save_result(result):

    if AI_OUTPUT_PATH.exists():

        existing = pd.read_csv(
            AI_OUTPUT_PATH
        )

        updated = pd.concat(
            [
                existing,
                pd.DataFrame([result])
            ],
            ignore_index=True
        )

    else:

        updated = pd.DataFrame(
            [result]
        )

    updated.to_csv(
        AI_OUTPUT_PATH,
        index=False
    )


# ============================================================
# INVESTIGATE ALL WITH RATE LIMITING
# ============================================================

def investigate_all(exceptions):

    previous = load_previous_results()

    successful_ids = set(
        previous["payment_id"].astype(str)
    )

    total = len(exceptions)

    processed = 0

    requests_this_minute = 0

    minute_start = time.time()

    for _, row in exceptions.iterrows():

        payment_id = str(
            row["payment_id"]
        )

        # ----------------------------------------------------
        # Skip already completed cases
        # ----------------------------------------------------

        if payment_id in successful_ids:

            print(
                f"Skipping {payment_id} "
                f"(already completed)"
            )

            continue

        # ----------------------------------------------------
        # Rate limiting
        # ----------------------------------------------------

        elapsed = time.time() - minute_start

        if elapsed >= 60:

            requests_this_minute = 0

            minute_start = time.time()

        if requests_this_minute >= 14:

            wait_time = 60 - elapsed

            print(
                f"\nRate limit safety pause: "
                f"{wait_time:.1f} seconds\n"
            )

            time.sleep(
                max(wait_time, 1)
            )

            requests_this_minute = 0

            minute_start = time.time()

        # ----------------------------------------------------
        # Investigate
        # ----------------------------------------------------

        processed += 1

        print(
            f"Investigating "
            f"{payment_id}..."
        )

        case = build_case(row)

        try:

            result = investigate(case)

            output = {

                "payment_id": payment_id,

                "exception": row["reason"],

                "ai_classification":
                    result["classification"],

                "explanation":
                    result["explanation"],

                "difference":
                    result["difference"],

                "recommended_action":
                    result["recommended_action"],

                "confidence":
                    result["confidence"]
            }

            save_result(output)

            successful_ids.add(
                payment_id
            )

            requests_this_minute += 1

            print(
                f"  ✓ "
                f"{result['classification']}"
            )

        except Exception as e:

            error_message = str(e)

            print(
                f"  ✗ Error: "
                f"{error_message}"
            )

            # ------------------------------------------------
            # If rate limited, wait and retry the SAME case
            # ------------------------------------------------

            if "429" in error_message:

                print(
                    "  Rate limit reached. "
                    "Waiting 60 seconds..."
                )

                time.sleep(60)

                requests_this_minute = 0

                minute_start = time.time()

                # Retry the same row
                try:

                    result = investigate(case)

                    output = {

                        "payment_id": payment_id,

                        "exception": row["reason"],

                        "ai_classification":
                            result["classification"],

                        "explanation":
                            result["explanation"],

                        "difference":
                            result["difference"],

                        "recommended_action":
                            result["recommended_action"],

                        "confidence":
                            result["confidence"]
                    }

                    save_result(output)

                    successful_ids.add(
                        payment_id
                    )

                    requests_this_minute += 1

                    print(
                        f"  ✓ Retry successful: "
                        f"{result['classification']}"
                    )

                except Exception as retry_error:

                    print(
                        f"  ✗ Retry failed: "
                        f"{retry_error}"
                    )

                    print(
                        "Stopping batch. "
                        "Run the script again later "
                        "to resume."
                    )

                    break

            else:

                print(
                    "Stopping batch because "
                    "an unexpected error occurred."
                )

                break

    print("\nInvestigation run complete.")

    final_results = load_previous_results()

    print(
        f"Successful investigations: "
        f"{len(final_results)} / {total}"
    )

    print(
        f"\nResults saved to:\n"
        f"{AI_OUTPUT_PATH}"
    )

    print(
        "\nClassification counts:"
    )

    if len(final_results) > 0:

        print(
            final_results[
                "ai_classification"
            ].value_counts()
        )


# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":

    print(
        "Loading reconciliation exceptions..."
    )

    exceptions = load_exceptions()

    print(
        f"Total exceptions: "
        f"{len(exceptions)}"
    )

    investigate_all(
        exceptions
    )

Loading reconciliation exceptions...
Total exceptions: 82
Investigating pay_0001...
  ✓ SETTLEMENT_DELAY
Investigating pay_0002...
  ✓ SETTLEMENT_DELAY
Investigating pay_0004...
  ✓ MISSING_BANK_RECORD
Investigating pay_0005...
  ✓ SETTLEMENT_DELAY
Investigating pay_0006...
  ✓ PARTIAL_REFUND
Investigating pay_0007...
  ✓ PARTIAL_REFUND
Investigating pay_0008...
  ✓ PARTIAL_REFUND
Investigating pay_0009...
  ✓ SETTLEMENT_DELAY
Investigating pay_0010...
  ✓ UNEXPLAINED_MISMATCH
Investigating pay_0012...
  ✓ UNEXPLAINED_MISMATCH
Investigating pay_0014...
  ✓ SETTLEMENT_DELAY
Investigating pay_0016...
  ✓ SETTLEMENT_DELAY
Investigating pay_0017...
  ✓ SETTLEMENT_DELAY
Investigating pay_0018...
  ✓ SETTLEMENT_DELAY
Investigating pay_0019...
  ✓ UNEXPLAINED_MISMATCH
Investigating pay_0020...
  ✓ MISSING_BANK_RECORD
Investigating pay_0021...
  ✓ SETTLEMENT_DELAY
Investigating pay_0022...
  ✓ PARTIAL_REFUND
Investigating pay_0023...
  ✓ SETTLEMENT_DELAY
Investigating pay_0024...
  ✓ MISSING_B

In [36]:
import pandas as pd

path = r"C:\Users\anshu\OneDrive\Desktop\prog\Projects\reconpilot\Tests\ai_investigation_results.csv"

results = pd.read_csv(path)

results["ai_classification"] = results["ai_classification"].replace(
    "MISSISSING_BANK_RECORD",
    "MISSING_BANK_RECORD"
)

results.to_csv(path, index=False)

print(results["ai_classification"].value_counts())

ai_classification
SETTLEMENT_DELAY        38
PARTIAL_REFUND          17
UNEXPLAINED_MISMATCH    16
MISSING_BANK_RECORD     11
Name: count, dtype: int64
